# Whisper LoRA — Indonesian Meeting ASR (Kaggle 2×T4)

Fallback for when the RTX 2060 (6 GB) is not enough. Kaggle gives two
T4s with 16 GB each, free, with a **9-hour session limit** and **30
GPU-hours per week**.

This notebook runs the *same* code as the local trainer — it clones the
repository and calls `ml/train/lora.py` with
`train/configs/whisper-turbo-id-kaggle-2xt4.yaml`. Nothing is
reimplemented here, because a notebook that reimplements the trainer is
a second trainer to keep correct.

## Before running

1. **Settings → Accelerator → GPU T4 ×2.**
2. **Settings → Internet → On** (needed to clone and to pull the base model).
3. Attach your dataset — see the *Data* cell for the two supported routes.
4. If a source is gated (GigaSpeech 2, Common Voice), add `HF_TOKEN` under
   **Add-ons → Secrets**.

> **Licence warning.** Training on GigaSpeech 2 means the resulting model
> must be released **non-commercially** (CC-BY-NC-4.0) with clear
> disclosure. `train/data.py::check_licences` detects this and records it
> in `train_metrics.json`. Read `ml/DATA_CARD.md` §3.3 first.


## 1. Environment


In [ ]:
!nvidia-smi
import torch

print(
    "torch",
    torch.__version__,
    "cuda",
    torch.cuda.is_available(),
    "devices",
    torch.cuda.device_count(),
)
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  [{i}] {p.name}  {p.total_memory / 1024**3:.1f} GiB")
# T4 is Turing: no bf16. The config keeps fp16 for this reason.
print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Pinned to what the configs were written against. bitsandbytes is
# what makes 8-bit base weights and 8-bit Adam work.
!pip install -q -U 'transformers>=4.44' 'peft>=0.13' 'accelerate>=0.34' \
    'datasets>=3.0' 'bitsandbytes>=0.43' 'evaluate>=0.4' 'soundfile>=0.12' \
    'librosa>=0.10' 'pyyaml>=6.0' 'jiwer'

## 2. Get the code

Replace the URL with your clone. A private repo needs a token in
Kaggle Secrets; a public one does not.


In [ ]:
import os, pathlib

REPO_URL = "https://github.com/trareon/transcribe.git"  # <-- edit
WORK = pathlib.Path("/kaggle/working")
REPO = WORK / "transcribe"

if not REPO.exists():
    !git clone --depth 1 {REPO_URL} {REPO}

ML = REPO / "ml"
os.chdir(ML)
# `ml/` is the project root: its packages (train, eval, align...) are
# imported by name, so it has to be on sys.path.
import sys

sys.path.insert(0, str(ML))
print(os.getcwd())
!ls

## 3. Data

Two routes. **A** is the normal one: the aligned shards produced by
`ml/align/pipeline.py` on the maintainer's machine, uploaded as a Kaggle
Dataset or pushed to a private HF dataset repo.

Either way the shard is an **audiofolder** (`data/*.wav` +
`metadata.jsonl`), which `datasets` reads with no loader script. That
matters: script-based datasets stopped working in `datasets` 3.x, which
is exactly why Common Voice and librivox-indonesia could not be read
for this project's benchmark.


In [ ]:
# --- Route A: Kaggle Dataset (Add Input -> your dataset) -------------
# Point TRAIN_SETS at the attached path.
TRAIN_SETS = ["/kaggle/input/trareon-align-shards/mk", "/kaggle/input/trareon-align-shards/dpr"]

# --- Route B: private HF dataset repo -------------------------------
# from huggingface_hub import snapshot_download
# from kaggle_secrets import UserSecretsClient
# token = UserSecretsClient().get_secret('HF_TOKEN')
# local = snapshot_download('your-org/trareon-align-shards',
#                           repo_type='dataset', token=token)
# TRAIN_SETS = [f'{local}/mk', f'{local}/dpr']

import pathlib

for path in TRAIN_SETS:
    p = pathlib.Path(path)
    ok = (p / "metadata.jsonl").exists()
    print(("OK  " if ok else "MISSING "), path)
    if not ok:
        print("     expected an audiofolder shard: data/*.wav + metadata.jsonl")

In [ ]:
# Make the gated-source token visible to `datasets` if present.
try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as error:
    print("no HF_TOKEN (fine unless a source is gated):", type(error).__name__)

## 4. Resolve the config

The checked-in config, with the data paths and output directory
overridden for Kaggle's filesystem. `load_config` rejects unknown keys,
so a typo fails here rather than three hours into a run.


In [ ]:
from train.config import load_config

config = load_config(
    "train/configs/whisper-turbo-id-kaggle-2xt4.yaml",
    train_sets=TRAIN_SETS,
    eval_sets=[],
    output_dir="/kaggle/working/out/whisper-turbo-id",
)

# Kaggle sessions die at 9 hours. Keep the run inside that, and rely on
# resume to continue in a second session: checkpoints land in
# /kaggle/working, which persists as the notebook's output.
config.max_steps = 2000
config.save_steps = 100

print("base        :", config.base_model)
print(
    "quantisation:",
    "8-bit" if config.load_in_8bit else ("4-bit" if config.load_in_4bit else "fp16"),
)
print(
    "batch       :",
    config.per_device_batch_size,
    "x",
    config.gradient_accumulation_steps,
    "=",
    config.effective_batch_size,
)
print("steps       :", config.max_steps)
print("output      :", config.output_dir)

## 5. Licence check — run it, read it

What the training sources allow decides what the model may be released
as. This is the same function the local trainer calls.


In [ ]:
from train.data import check_licences

verdict = check_licences(config.train_sets)
print(verdict.summary())
print()
print("commercial use allowed :", verdict.commercial_use_allowed)
print("recommended licence    :", verdict.recommended_licence)

## 6. VRAM dry run

A few real steps on synthetic batches, reporting peak VRAM. On a 16 GB
T4 this config should fit with plenty of headroom; run it anyway, because
it is also the fastest way to find a broken `bitsandbytes`.


In [ ]:
from train.dry_run_memory import measure

report = measure(config, steps=3)
print(report.verdict())
report.as_dict()

## 7. Train

Resumable. If the session dies, re-run this cell in a new session with
the same `output_dir` attached and it continues from the last checkpoint.


In [ ]:
from train.lora import train

metrics = train(config)
metrics

## 8. Merge the adapter and export to GGML

The GGML conversion clones `whisper.cpp` and builds its quantiser, which
takes a few minutes. Skip it here and do it locally if the session is
running short — the adapter is the valuable output.


In [ ]:
from train.merge_export import merge, to_ggml, write_model_card

ADAPTER = f"{config.output_dir}/adapter"
EXPORT = "/kaggle/working/out/export"

merged = merge(ADAPTER, EXPORT)
print("merged:", merged)

In [ ]:
# Optional, and slow. Set to False to skip.
DO_GGML = True
ggml = to_ggml(merged, quantize="q5_0") if DO_GGML else None
print("ggml:", ggml)

write_model_card(EXPORT, base_model=config.base_model, adapter=ADAPTER, ggml=ggml, metrics=metrics)
!ls -la {EXPORT}

## 9. Measure it

Whatever comes out of here is unmeasured until it is scored on the
benchmark's test sets. Scoring the *merged* model with the transformers
runner works on Kaggle; the GGML path needs `rust_core`'s `wer_bench`,
which is a Rust build and belongs on the local machine.


In [ ]:
from collect.hf_sets import FLEURS_ID, fetch
from eval.harness import TestSet, render_markdown, run_benchmark
from eval.runners import TransformersRunner

# FLEURS is public and small. Read speech, so a good score here proves
# much less than it looks - see BENCHMARK.md.
manifest = fetch(FLEURS_ID, "/kaggle/working/data/fleurs-id", clips=40)

test_sets = [
    TestSet(
        name="fleurs-id",
        manifest=manifest,
        domain="ucapan baca, satu penutur",
        speech_kind="baca",
        licence_note="CC-BY 4.0",
    )
]

runners = [
    TransformersRunner(model_id=config.base_model, device="cuda", name="basis"),
    TransformersRunner(model_id=str(merged), device="cuda", name="fine-tune"),
]
result = run_benchmark(runners, test_sets, limit=40)
print(render_markdown(result, test_sets))

## 10. Keep the output

`/kaggle/working` is saved as the notebook's output when the session
ends — but it has a size cap, so delete the merged fp32 intermediates
and keep the adapter (tens of MB) plus the GGML.


In [ ]:
!du -sh /kaggle/working/out/* 2>/dev/null
# The adapter is the thing worth keeping; the merged model can be
# rebuilt from it in minutes.
!ls -la {ADAPTER}